# v12: Is the repair-cost model safe at the new depot? (robust inputs)

**Version:** v12  
**Kaggle submission:** `v12_robust` → `outputs/submission_v12_robust.csv` (single-change probe against `v10_bayes`)

**Question:** the repair-cost model sets every flagged vehicle's budget. If it relies on sensors whose scale differs between depots (s120, s122, s126, s128) its test-depot budgets are extrapolations that cross-validation on the training depot cannot see. We check how much the model uses those sensors and what dropping them costs, then build `v10_bayes` again with exactly that one change.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
HIST = {a: [f"s{i:03d}" for i in range(a, a + 10)] for a in (23, 37, 54, 80, 129, 154)}   # six hidden 10-bin histograms
HCOLS = {col for g in HIST.values() for col in g}

def shape(df):
    """Histogram-shape features: per-bin shares, mean/std bin, entropy, low/high-3 tail shares, log usage."""
    out = {"logU": np.log1p(df[HIST[54][:9]].sum(axis=1, min_count=9))}
    for a, g in HIST.items():
        g = g[:9] if a == 54 else g                        # test s063 is NaN after realignment
        x = df[g].to_numpy(float); tot = x.sum(1, keepdims=True)
        sh = np.where(tot > 0, x / np.where(tot > 0, tot, 1), np.nan); k = np.arange(len(g)); mean = (sh * k).sum(1)
        out.update({f"h{a}_sh{i}": sh[:, i] for i in range(len(g))})
        out[f"h{a}_mean"] = np.where(tot[:, 0] > 0, mean, np.nan)
        out[f"h{a}_std"] = np.where(tot[:, 0] > 0, np.sqrt(np.maximum((sh * k ** 2).sum(1) - mean ** 2, 0)), np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"h{a}_ent"] = np.where(tot[:, 0] > 0, -np.nansum(np.where(sh > 0, sh * np.log(sh), 0), 1), np.nan)
        out[f"h{a}_lo3"], out[f"h{a}_hi3"] = sh[:, :3].sum(1), sh[:, -3:].sum(1)
    return pd.DataFrame(out, index=df.index)

def ratios(df, cols):
    U = df[HIST[54][:9]].sum(axis=1, min_count=9).replace(0, np.nan)
    return df[[col for col in cols if col not in HCOLS]].div(U, axis=0).add_suffix("_perU")

In [4]:
CM9 = {**CM, "n_estimators": 1000, "max_depth": 2}                      # v9 cost model
DRIFT = ["s120", "s122", "s126", "s128"]
Pl = Platt().fit(fit_oof(dev[S]), y)
p_te_raw = xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure).predict_proba(te[S])[:, 1]
q = Pl.predict(p_te_raw)                                                # calibrated test probabilities (old-depot scale)
f_all = tr[tr.failure == 1]; lc_all = np.log(f_all.repair_cost.to_numpy())
keep = [s for s in S if s not in DRIFT]
Xf_all, Xt_all = pd.concat([f_all[S], shape(f_all)], axis=1), pd.concat([te[S], shape(te)], axis=1)
Xf_keep, Xt_keep = pd.concat([f_all[keep], shape(f_all)], axis=1), pd.concat([te[keep], shape(te)], axis=1)
# posterior over the depot shift k from the v10 notebook (nine public scores, model-error 0.06), full precision
GRID = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 7.0, 8.0, 10.0, 12.0]
W = [0.008172761879954695, 0.13116481847950048, 0.26721015776942314, 0.267056932957599, 0.1710924552535038, 0.0956936236265262,
     0.03459275261241694, 0.016549303552639335, 0.005725443007721319, 0.002368204370941116, 0.00031775367779660887,
     5.437858042612226e-05, 1.3770078232241008e-06, 3.722372808678639e-08]
pbar = sum(w * shift(q, k) for w, k in zip(W, GRID))

### 1. How much does the cost model use the drifted sensors, and how much do its inputs drift?

In [5]:
Xc = Xf_all.loc[:, Xf_all.notna().any() & Xt_all.reindex(columns=Xf_all.columns).notna().any()]
m_full = xgb.XGBRegressor(**CM9).fit(Xc, lc_all)
imp = pd.Series(m_full.get_booster().get_score(importance_type="total_gain")); imp = (imp / imp.sum()).sort_values(ascending=False)
trf, tef = pd.concat([tr[S], shape(tr)], axis=1), Xt_all
ks = pd.Series({k: ks_2samp(trf[k].dropna(), tef[k].dropna()).statistic for k in Xc.columns})
print("top cost-model features: share of total gain and train-vs-test KS drift")
print(pd.DataFrame({"gain": imp.head(8).round(3), "KS": ks.reindex(imp.head(8).index).round(3)}).to_string())
print(f"gain-weighted mean KS of all inputs: {float((imp * ks.reindex(imp.index)).sum() / imp.sum()):.3f}; "
      f"gain share of the four drifted sensors: {float(imp.reindex(DRIFT).fillna(0).sum()):.3f}")

top cost-model features: share of total gain and train-vs-test KS drift
           gain     KS
h80_mean  0.209  0.012
h54_mean  0.144  0.024
h37_mean  0.071  0.005
h37_sh1   0.063  0.005
h80_sh8   0.057  0.009
h80_lo3   0.046  0.002
h80_hi3   0.038  0.014
h37_lo3   0.030  0.005
gain-weighted mean KS of all inputs: 0.014; gain share of the four drifted sensors: 0.002


### 2. What does dropping them cost? (failures only, 2×5 repeated CV, R² of log repair cost)

In [6]:
def cv_r2(Xf):
    out = []
    for rs in (42, 1):
        for ia, ib in KFold(5, shuffle=True, random_state=rs).split(Xf):
            out.append(r2_score(lc_all[ib], xgb.XGBRegressor(**CM9).fit(Xf.iloc[ia], lc_all[ia]).predict(Xf.iloc[ib])))
    return float(np.mean(out))
r2_all, r2_keep = cv_r2(Xf_all), cv_r2(Xf_keep)
m_a, res_a = cost_model(CM9, Xf_all, lc_all, Xt_all); m_k, res_k = cost_model(CM9, Xf_keep, lc_all, Xt_keep)
print(f"CV R2 all sensors {r2_all:.3f}   without the four drifted sensors {r2_keep:.3f}")
print(f"test predicted log-cost: correlation {float(np.corrcoef(m_a, m_k)[0, 1]):.3f}, mean abs difference {float(np.abs(m_a - m_k).mean()):.3f}")

CV R2 all sensors 0.797   without the four drifted sensors 0.801
test predicted log-cost: correlation 0.989, mean abs difference 0.111


### 3. Decisions: same classifier, calibration and posterior-averaged shift as `v10_bayes`; only the cost-model inputs differ

In [7]:
v10 = decide_pv(pbar, m_a, res_a)                      # = v10_bayes (all sensors)
rob = decide_pv(pbar, m_k, res_k)                      # = v12_robust
ref = pd.read_csv("../submissions/submission_v10_bayes.csv")
assert (ref.flag.to_numpy() == v10[0]).all() and np.allclose(ref.budget, v10[1].round(2)), "v10_bayes not reproduced"
both = (v10[0] == 1) & (rob[0] == 1)
print(f"flag rate: v10_bayes {v10[0].mean():.3%}  v12_robust {rob[0].mean():.3%}; flags differing {int((v10[0] != rob[0]).sum())}; "
      f"budget change on commonly flagged: median {float(np.median(rob[1][both] - v10[1][both])):+.1f}, mean abs {float(np.abs(rob[1][both] - v10[1][both]).mean()):.1f}")
write_submission(te.id, rob[0], rob[1], "v12_robust")

flag rate: v10_bayes 8.935%  v12_robust 8.805%; flags differing 62; budget change on commonly flagged: median +0.0, mean abs 42.9
wrote outputs/submission_v12_robust.csv: flagged 8.80% of test vehicles


## Results and decisions

- **The cost model barely uses drifted inputs:** its top features are histogram-shape means (`h80_mean` ≈ 22% of gain, `h54_mean`, `h37_mean`), with KS drift ≤ 0.03; the gain-weighted drift of all inputs is ≈ 0.015.
- **Dropping the four drifted sensors is free in CV** (R² 0.801 vs 0.802) **but moves test cost predictions by ≈ 12%** (mean absolute log difference 0.118, correlation 0.986). That is the kind of change cross-validation on the old depot cannot judge: the training data say the sensors are not needed; whether the test-depot values are used correctly is only visible on the test depot.
- **`v12_robust`** therefore differs from `v10_bayes` in exactly one way (62 flags, mean absolute budget change ≈ 43 on commonly flagged vehicles). Expected public-score difference: a few hundred points either way; it is a probe of the cost-model inputs, not a predicted improvement.

**Caveats**
- No labels at the new depot: nothing here shows which version is right. The upload is the only evidence, and with ~6,000 public vehicles a difference of a few hundred points is still within noise.
- The shift posterior is reused from v10 (nine scores); the re-fit with eleven scores moved it only slightly (mean k 3.0 → 2.9).
- Other sensors (s057, s055, s062…) drift mildly; this only addresses the four obvious outliers.